# NBA Playoff Prediction — Data Collection & Preprocessing
**Author:** Anthony  
**Project:** NBA Playoff Matchup Predictor  

---

## Datasets Used

| Dataset | Source | What We Use It For |
|---|---|---|
| NBA Games (nathanlauga) | [Kaggle](https://www.kaggle.com/datasets/nathanlauga/nba-games) | Game-by-game results, box scores 2004–present |
| NBA Database (wyattowalsh) | [Kaggle](https://www.kaggle.com/datasets/wyattowalsh/basketball) | Comprehensive stats, advanced metrics, 1946–present |
| NBA Stats 1947–present (sumitrodatta) | [Kaggle](https://www.kaggle.com/datasets/sumitrodatta/nba-aba-baa-stats) | Team/player season averages |
| FiveThirtyEight NBA ELO | [GitHub](https://raw.githubusercontent.com/fivethirtyeight/data/master/nba-elo/nbaallelo.csv) | Pre-computed ELO ratings as a baseline feature |
| Updated NBA ELO (Neil Paine) | [GitHub](https://github.com/Neil-Paine-1/NBA-elo) | More recent ELO updates post-538 |
| ProSportsTransactions | [Website](https://www.prosportstransactions.com) | Injury/rest data |

### How to Download the Kaggle Datasets
1. Install the Kaggle CLI: `pip install kaggle`
2. Set up your API key from https://www.kaggle.com/settings → API → Create New Token
3. Place `kaggle.json` in `~/.kaggle/`
4. Run the cells below or manually download and place CSVs in a `data/` folder

```bash
kaggle datasets download -d nathanlauga/nba-games -p data/ --unzip
kaggle datasets download -d wyattowalsh/basketball -p data/ --unzip
kaggle datasets download -d sumitrodatta/nba-aba-baa-stats -p data/ --unzip
```

In [ ]:
import pandas as pd
import numpy as np
import warnings
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)

DATA_DIR = 'data/'  # <-- change this if your CSVs are in a different folder
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs('output/', exist_ok=True)

print('Libraries loaded.')

---
## Step 1: Load Raw Data

In [ ]:
# ── Load from nathanlauga/nba-games ──────────────────────────────────────────
# games.csv: one row per game (team-level box score)
# games_details.csv: one row per player per game
# ranking.csv: daily standings

games = pd.read_csv(f'{DATA_DIR}games.csv')
games_details = pd.read_csv(f'{DATA_DIR}games_details.csv')
ranking = pd.read_csv(f'{DATA_DIR}ranking.csv')
teams = pd.read_csv(f'{DATA_DIR}teams.csv')

print(f'games:         {games.shape}')
print(f'games_details: {games_details.shape}')
print(f'ranking:       {ranking.shape}')
print(f'teams:         {teams.shape}')
games.head(3)

In [ ]:
# ── Load FiveThirtyEight ELO (direct URL — no Kaggle account needed) ─────────
elo_538 = pd.read_csv(
    'https://raw.githubusercontent.com/fivethirtyeight/data/master/nba-elo/nbaallelo.csv'
)
print(f'538 ELO dataset: {elo_538.shape}')
print(elo_538.columns.tolist())
elo_538.head(3)

---
## Step 2: Initial Exploration & Sanity Checks

In [ ]:
print('=== games.csv columns ===')
print(games.dtypes)
print()
print('=== Missing value % ===')
missing = (games.isnull().sum() / len(games) * 100).round(2)
print(missing[missing > 0])
print()
print('=== Season range ===')
print(games['SEASON'].value_counts().sort_index())

In [ ]:
# Check playoff flag — nathanlauga dataset uses GAME_STATUS_TEXT or similar
# Let's see what unique values exist so we know how to filter playoffs
for col in ['GAME_STATUS_TEXT', 'HOME_TEAM_WINS', 'SEASON']:
    if col in games.columns:
        print(f'{col}: {games[col].unique()[:10]}')

---
## Step 3: Cleaning — games.csv

In [ ]:
def clean_games(df):
    """
    Cleans the raw games.csv from nathanlauga/nba-games.
    - Parses dates
    - Filters to 2000 season onwards
    - Drops games with missing score data
    - Standardizes column names
    - Separates regular season and playoffs
    """
    df = df.copy()

    # ── 1. Parse and sort by date ─────────────────────────────────────────────
    df['GAME_DATE_EST'] = pd.to_datetime(df['GAME_DATE_EST'])
    df = df.sort_values('GAME_DATE_EST').reset_index(drop=True)

    # ── 2. Filter to modern era (2000 season onwards) ─────────────────────────
    df = df[df['SEASON'] >= 2000].copy()

    # ── 3. Drop rows where either team's score is missing ─────────────────────
    score_cols = ['PTS_home', 'PTS_away']
    before = len(df)
    df = df.dropna(subset=score_cols)
    print(f'Dropped {before - len(df)} rows with missing scores. Remaining: {len(df)}')

    # ── 4. Add explicit win/loss columns ─────────────────────────────────────
    # HOME_TEAM_WINS = 1 if home won, 0 if away won
    df['HOME_WIN'] = df['HOME_TEAM_WINS'].astype(int)
    df['AWAY_WIN'] = 1 - df['HOME_WIN']

    # ── 5. Calculate point differential ──────────────────────────────────────
    df['HOME_MARGIN'] = df['PTS_home'] - df['PTS_away']

    # ── 6. Separate regular season vs playoffs ────────────────────────────────
    # nathanlauga: GAME_ID starting with '004' = playoffs, '002' = regular season
    # (This is the standard NBA Stats API convention)
    df['GAME_TYPE'] = df['GAME_ID'].astype(str).str[2:5].map({
        '002': 'Regular',
        '004': 'Playoff',
        '001': 'Preseason',
        '003': 'AllStar'
    }).fillna('Other')

    print(f"\nGame type breakdown:\n{df['GAME_TYPE'].value_counts()}")
    return df

games_clean = clean_games(games)
games_clean.head(3)

In [ ]:
# Split into regular season and playoff subsets
regular_season = games_clean[games_clean['GAME_TYPE'] == 'Regular'].copy()
playoffs = games_clean[games_clean['GAME_TYPE'] == 'Playoff'].copy()

print(f'Regular season games: {len(regular_season)}')
print(f'Playoff games:        {len(playoffs)}')
print(f'Playoff seasons:      {sorted(playoffs["SEASON"].unique())}')

---
## Step 4: Dataset Doubling (T1 / T2 Perspective Swap)

Adapted from our March ML Mania notebook. Creates a symmetric training set where T1 is always "the team we're predicting for" — doubling the dataset so the model sees both perspectives of every game.

In [ ]:
def build_base_features(df):
    """
    Converts the nathanlauga home/away format into the symmetric T1/T2 doubling format,
    adapted from our March ML Mania pipeline.

    Input columns expected (from games_clean):
        HOME_TEAM_ID, VISITOR_TEAM_ID, GAME_DATE_EST, SEASON, GAME_TYPE, GAME_ID,
        PTS_home, PTS_away, HOME_MARGIN, HOME_WIN,
        FG_PCT_home, FG_PCT_away, FT_PCT_home, FT_PCT_away,
        FG3_PCT_home, FG3_PCT_away, AST_home, AST_away,
        REB_home, REB_away
    """
    df = df.copy()

    # The box score stat columns available in this dataset
    stat_cols = ['PTS', 'FG_PCT', 'FT_PCT', 'FG3_PCT', 'AST', 'REB']

    # ── Home team perspective (T1 = Home) ─────────────────────────────────────
    df_home = pd.DataFrame()
    df_home['Season']      = df['SEASON']
    df_home['GameDate']    = df['GAME_DATE_EST']
    df_home['GameID']      = df['GAME_ID']
    df_home['GameType']    = df['GAME_TYPE']
    df_home['T1_TeamID']   = df['HOME_TEAM_ID']
    df_home['T2_TeamID']   = df['VISITOR_TEAM_ID']
    df_home['T1_Loc']      = 1   # Home
    for col in stat_cols:
        df_home[f'T1_{col}'] = df[f'{col}_home']
        df_home[f'T2_{col}'] = df[f'{col}_away']
    df_home['Target_Win']  = df['HOME_WIN']

    # ── Away team perspective (T1 = Away) ─────────────────────────────────────
    df_away = pd.DataFrame()
    df_away['Season']      = df['SEASON']
    df_away['GameDate']    = df['GAME_DATE_EST']
    df_away['GameID']      = df['GAME_ID']
    df_away['GameType']    = df['GAME_TYPE']
    df_away['T1_TeamID']   = df['VISITOR_TEAM_ID']
    df_away['T2_TeamID']   = df['HOME_TEAM_ID']
    df_away['T1_Loc']      = -1  # Away
    for col in stat_cols:
        df_away[f'T1_{col}'] = df[f'{col}_away']
        df_away[f'T2_{col}'] = df[f'{col}_home']
    df_away['Target_Win']  = 1 - df['HOME_WIN']

    combined = pd.concat([df_home, df_away], ignore_index=True)
    combined = combined.sort_values(['Season', 'GameDate']).reset_index(drop=True)

    return combined

regular_data = build_base_features(regular_season)
playoff_data = build_base_features(playoffs)

print(f'Regular (doubled): {regular_data.shape}')
print(f'Playoffs (doubled): {playoff_data.shape}')
regular_data.head(3)

---
## Step 5: Season-Level Feature Engineering

For each team and season, compute the features we'll merge onto matchups — same logic as March ML Mania but using NBA box score columns.

In [ ]:
def get_season_averages(regular_df):
    """
    Calculates season-long offensive and defensive averages per team.
    Mirrors the NCAA pipeline but uses NBA stat columns.
    """
    stat_cols = ['PTS', 'FG_PCT', 'FT_PCT', 'FG3_PCT', 'AST', 'REB']
    cols_to_avg = [f'T1_{c}' for c in stat_cols] + [f'T2_{c}' for c in stat_cols]

    season_avgs = (
        regular_df
        .groupby(['Season', 'T1_TeamID'])[cols_to_avg]
        .mean()
        .reset_index()
        .rename(columns={'T1_TeamID': 'TeamID'})
    )

    rename = {}
    for c in stat_cols:
        rename[f'T1_{c}'] = f'Avg_{c}'        # Offensive production
        rename[f'T2_{c}'] = f'Avg_Opp_{c}'    # What they allowed opponents
    return season_avgs.rename(columns=rename)

season_averages = get_season_averages(regular_data)
print(season_averages.shape)
season_averages.head(3)

In [ ]:
def get_advanced_season_stats(regular_df):
    """
    Calculates Offensive and Defensive Efficiency (points per 100 possessions)
    and Net Rating for each team.

    NOTE: nathanlauga doesn't include FGA/FTA/OR/TO directly, so we estimate
    possessions from what's available. If you have the wyattowalsh dataset,
    swap in the full possession formula: FGA - OR + TO + 0.475*FTA
    """
    df = regular_df.copy()

    # Rough possession proxy using just field goal attempts and free throws
    # (Replace with full formula once wyattowalsh data is merged in)
    # Efficiency here is points per game — upgrade to /100 possessions later

    team_stats = []
    for (season, team_id), group in df.groupby(['Season', 'T1_TeamID']):
        avg_pts_scored  = group['T1_PTS'].mean()
        avg_pts_allowed = group['T2_PTS'].mean()
        net_rating      = avg_pts_scored - avg_pts_allowed
        win_pct         = group['Target_Win'].mean()

        team_stats.append({
            'Season':       season,
            'TeamID':       team_id,
            'Avg_PTS_For':  avg_pts_scored,
            'Avg_PTS_Ag':   avg_pts_allowed,
            'Net_Rating':   net_rating,
            'Win_Pct':      win_pct,
        })

    return pd.DataFrame(team_stats)

advanced_stats = get_advanced_season_stats(regular_data)
print(advanced_stats.shape)
advanced_stats.head(3)

In [ ]:
def calculate_elo(doubled_df, base_elo=1500, k_factor=20, hca=100):
    """
    Calculates end-of-season ELO for each team.
    Uses the FiveThirtyEight margin-of-victory multiplier.
    Directly ported from our March ML Mania pipeline.
    """
    # Undo doubling: keep only rows where T1 won (one row per game)
    unique_games = doubled_df[doubled_df['Target_Win'] == 1].copy()
    unique_games = unique_games.sort_values(['Season', 'GameDate'])

    final_elos = []

    for season, season_df in unique_games.groupby('Season'):
        teams = set(season_df['T1_TeamID']) | set(season_df['T2_TeamID'])
        elo_dict = {team: base_elo for team in teams}

        for _, row in season_df.iterrows():
            w_team = row['T1_TeamID']
            l_team = row['T2_TeamID']
            w_pts  = row['T1_PTS']
            l_pts  = row['T2_PTS']
            w_loc  = row['T1_Loc']   # 1=Home, -1=Away

            w_elo = elo_dict[w_team]
            l_elo = elo_dict[l_team]

            # Apply home court adjustment
            w_elo_adj = w_elo + (hca if w_loc == 1 else 0)
            l_elo_adj = l_elo + (hca if w_loc == -1 else 0)

            expected_win = 1.0 / (1.0 + 10.0 ** ((l_elo_adj - w_elo_adj) / 400.0))

            mov = w_pts - l_pts
            elo_diff = w_elo - l_elo
            mov_multiplier = np.log(max(mov, 1) + 1.0) * (2.2 / ((elo_diff * 0.001) + 2.2))
            mov_multiplier = max(mov_multiplier, 0.1)

            change = k_factor * mov_multiplier * (1.0 - expected_win)
            elo_dict[w_team] = w_elo + change
            elo_dict[l_team] = l_elo - change

        for team, elo in elo_dict.items():
            final_elos.append({'Season': season, 'TeamID': team, 'End_Season_Elo': elo})

    return pd.DataFrame(final_elos)

elo_df = calculate_elo(regular_data)
print(elo_df.shape)
elo_df.sort_values('End_Season_Elo', ascending=False).head(10)

In [ ]:
def calculate_win_percentages(doubled_df):
    """
    Calculates Overall, Home, and Away win percentages.
    Missing location records are imputed with the overall win pct.
    """
    overall = (
        doubled_df
        .groupby(['Season', 'T1_TeamID'])['Target_Win']
        .mean().reset_index()
        .rename(columns={'T1_TeamID': 'TeamID', 'Target_Win': 'Overall_Win_Pct'})
    )
    home = (
        doubled_df[doubled_df['T1_Loc'] == 1]
        .groupby(['Season', 'T1_TeamID'])['Target_Win']
        .mean().reset_index()
        .rename(columns={'T1_TeamID': 'TeamID', 'Target_Win': 'Home_Win_Pct'})
    )
    away = (
        doubled_df[doubled_df['T1_Loc'] == -1]
        .groupby(['Season', 'T1_TeamID'])['Target_Win']
        .mean().reset_index()
        .rename(columns={'T1_TeamID': 'TeamID', 'Target_Win': 'Away_Win_Pct'})
    )

    result = overall.merge(home, on=['Season', 'TeamID'], how='left')
    result = result.merge(away, on=['Season', 'TeamID'], how='left')

    result['Home_Win_Pct'] = result['Home_Win_Pct'].fillna(result['Overall_Win_Pct'])
    result['Away_Win_Pct'] = result['Away_Win_Pct'].fillna(result['Overall_Win_Pct'])

    return result

win_pct_df = calculate_win_percentages(regular_data)
win_pct_df.head(3)

In [ ]:
def get_recent_form(regular_df, last_n=10):
    """
    Win percentage over the last N games of the regular season per team.
    Captures 'hot/cold' momentum heading into the playoffs.
    """
    recent = (
        regular_df
        .sort_values(['Season', 'GameDate'])
        .groupby(['Season', 'T1_TeamID'])
        .tail(last_n)
        .groupby(['Season', 'T1_TeamID'])['Target_Win']
        .mean().reset_index()
        .rename(columns={'T1_TeamID': 'TeamID', 'Target_Win': f'Last{last_n}_Win_Pct'})
    )
    return recent

recent_form = get_recent_form(regular_data, last_n=10)
recent_form.head(3)

In [ ]:
def get_rest_days(games_df):
    """
    NBA-specific feature: days of rest before each game.
    Works on the NON-doubled (home/away) games dataframe.
    Returns rest days for both home and away teams.
    """
    df = games_df[['GAME_ID', 'GAME_DATE_EST', 'SEASON', 'HOME_TEAM_ID', 'VISITOR_TEAM_ID']].copy()
    df = df.sort_values('GAME_DATE_EST')

    # Build a long-form table: one row per team per game
    home_rows = df[['GAME_ID', 'GAME_DATE_EST', 'SEASON', 'HOME_TEAM_ID']].rename(
        columns={'HOME_TEAM_ID': 'TeamID'})
    away_rows = df[['GAME_ID', 'GAME_DATE_EST', 'SEASON', 'VISITOR_TEAM_ID']].rename(
        columns={'VISITOR_TEAM_ID': 'TeamID'})

    all_games = pd.concat([home_rows, away_rows]).sort_values('GAME_DATE_EST')

    # Calculate days since last game for each team
    all_games['Prev_Game_Date'] = all_games.groupby('TeamID')['GAME_DATE_EST'].shift(1)
    all_games['Rest_Days'] = (
        pd.to_datetime(all_games['GAME_DATE_EST']) -
        pd.to_datetime(all_games['Prev_Game_Date'])
    ).dt.days.fillna(7)  # Impute first game of season with 7 days

    # Cap rest days at 14 (anything longer is a long break / start of season)
    all_games['Rest_Days'] = all_games['Rest_Days'].clip(upper=14)

    return all_games[['GAME_ID', 'TeamID', 'Rest_Days']]

rest_days = get_rest_days(regular_season)  # or pass playoffs for playoff rest
print(rest_days.shape)
rest_days.head(5)

---
## Step 6: Build Master Stats Table

In [ ]:
# Merge all team-level features into one master table
master_stats = advanced_stats.copy()
master_stats = master_stats.merge(elo_df,         on=['Season', 'TeamID'], how='inner')
master_stats = master_stats.merge(win_pct_df,     on=['Season', 'TeamID'], how='inner')
master_stats = master_stats.merge(season_averages,on=['Season', 'TeamID'], how='inner')
master_stats = master_stats.merge(recent_form,    on=['Season', 'TeamID'], how='left')

# Impute missing recent form with overall win pct
master_stats['Last10_Win_Pct'] = master_stats['Last10_Win_Pct'].fillna(master_stats['Win_Pct'])

print(f'Master stats shape: {master_stats.shape}')
print(f'Columns: {master_stats.columns.tolist()}')
master_stats.head(3)

---
## Step 7: Map Features onto Matchups (Training Set)

In [ ]:
def map_features_to_matchups(matchup_df, stats_df):
    """
    Merges master stats onto both T1 and T2 and computes differentials.
    Directly adapted from March ML Mania pipeline.
    """
    df = matchup_df.copy()
    feature_cols = [c for c in stats_df.columns if c not in ['Season', 'TeamID', 'Win_Pct']]

    # Merge T1
    df = df.merge(
        stats_df, left_on=['Season', 'T1_TeamID'], right_on=['Season', 'TeamID'], how='left'
    ).drop(columns=['TeamID'])
    df = df.rename(columns={c: f'T1_{c}' for c in feature_cols})

    # Merge T2
    df = df.merge(
        stats_df, left_on=['Season', 'T2_TeamID'], right_on=['Season', 'TeamID'], how='left'
    ).drop(columns=['TeamID'])
    df = df.rename(columns={c: f'T2_{c}' for c in feature_cols})

    # Compute differentials
    for feat in feature_cols:
        df[f'{feat}_Diff'] = df[f'T1_{feat}'] - df[f'T2_{feat}']

    return df

# Build training set from regular season games (predicting each game outcome)
train_dataset = map_features_to_matchups(regular_data, master_stats)

# Drop raw in-game stats (data leakage — model can't see these at prediction time)
leak_cols = ['T1_PTS', 'T2_PTS', 'T1_FG_PCT', 'T2_FG_PCT',
             'T1_FT_PCT', 'T2_FT_PCT', 'T1_FG3_PCT', 'T2_FG3_PCT',
             'T1_AST', 'T2_AST', 'T1_REB', 'T2_REB', 'T1_Loc']
train_dataset = train_dataset.drop(columns=[c for c in leak_cols if c in train_dataset.columns])

X_train = train_dataset.drop(columns=['Target_Win'])
y_train = train_dataset['Target_Win']

print(f'Training set: {X_train.shape}')
print(f'Missing values in X_train: {X_train.isnull().sum().sum()}')
X_train.head(3)

---
## Step 8: Build Playoff Test Set

For the playoff set, we don't have future data — we use the prior regular season's master stats to predict each playoff game.

In [ ]:
# Shift season by 1: use current season regular stats to predict current season playoffs
# (They play in the same year — no shift needed; just make sure master_stats are pre-playoff)

playoff_test = map_features_to_matchups(playoff_data, master_stats)
playoff_test = playoff_test.drop(columns=[c for c in leak_cols if c in playoff_test.columns])

X_playoff = playoff_test.drop(columns=['Target_Win'])
y_playoff = playoff_test['Target_Win']

print(f'Playoff test set: {X_playoff.shape}')
print(f'Playoff seasons covered: {sorted(playoff_test["Season"].unique())}')

---
## Step 9: Save Clean Datasets

In [ ]:
master_stats.to_csv('output/master_stats.csv', index=False)
X_train.to_csv('output/X_train.csv', index=False)
y_train.to_csv('output/y_train.csv', index=False)
X_playoff.to_csv('output/X_playoff.csv', index=False)
y_playoff.to_csv('output/y_playoff.csv', index=False)

print('All cleaned datasets saved to output/')
print(f'  master_stats.csv  — {master_stats.shape}')
print(f'  X_train.csv       — {X_train.shape}')
print(f'  y_train.csv       — {y_train.shape}')
print(f'  X_playoff.csv     — {X_playoff.shape}')
print(f'  y_playoff.csv     — {y_playoff.shape}')

---
## Notes for Olivia & Benjamin (Modeling)

The files in `output/` are ready to plug into the modeling notebook:

- **X_train / y_train** — regular season games with team-level differential features. Use these to train the model.
- **X_playoff / y_playoff** — playoff games with the same features. Use these to validate that the model generalizes to the postseason.
- **master_stats** — per-team per-season feature table. Use this to generate predictions for future playoff matchups.

Key features in the differential columns:
- `End_Season_Elo_Diff` — ELO rating gap (most predictive single feature from March ML Mania)
- `Net_Rating_Diff` — points scored minus allowed, per game
- `Overall_Win_Pct_Diff` — regular season win % gap
- `Last10_Win_Pct_Diff` — recent form heading into the playoffs
- `Avg_PTS_For_Diff`, `Avg_PTS_Ag_Diff` — offensive and defensive scoring gaps
- `Away_Win_Pct_Diff` — road performance gap (relevant since higher seeds have home court advantage)

**Recommended validation:** Leave-one-season-out (walk-forward by year) — same as March ML Mania notebook.